<a href="https://colab.research.google.com/github/ajoven10/working-with-alphagenome/blob/notebooks/Relevancia_nucleotidos_shap_rna_seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Highlighting important regions with in silico mutagenesis: SHAP vs ISM AlphaGenome

## Introduction

This notebook performs an interpretability analysis on a specific DNA sequence using the AlphaGenome API, employing two different methods: In-Silico Mutagenesis (ISM) and SHAP (SHapley Additive exPlanations).

The ultimate goal is to visualize and compare the contribution of each nucleotide position to the model's prediction.

Script Summary
1. **Setup and Data Loading**
Initialization: Imports necessary libraries (alphagenome, matplotlib, numpy, shap, etc.) and sets up the AlphaGenome API key.

Genome Load: Loads the UCSC hg38 genome FASTA file into memory using Biopython's SeqIO.

Target Interval: Defines a 2KB DNA sequence (sequence_interval) centered around a region on chr20 (3,753,000 to 3,753,400) to provide context for the model.

2. **In-Silico Mutagenesis (ISM)** Calculation
ISM Definition: Defines a central 256-base region (ism_interval) within the 2KB sequence for mutation.

Scoring: Configures a CenterMaskScorer to measure the difference in the DNASE accessibility prediction for the K562 cell line (EFO:0002067) when each base in the 256-base region is mutated.

Result: The dna_model.score_ism_variants is called to generate variant scores, which are then processed by ism.ism_matrix. The final Alphagenome contribution (resultado_alphagenome) is calculated as the sum of contributions across all base types for each position.

3. **SHAP Value Calculation (Model Interpretation)**
Background Data Generation:

Random Sampling: 5000 random 256-base sequences are extracted from chr20.

Padding/One-Hot: These sequences are padded to 2048 bases (dna_len) and one-hot encoded to create the background_data_flat.

K-Means: The background data is reduced using K-Means clustering (K=5) to select a representative sample (background_data_sample) for the SHAP explainer.

Prediction Function: A predict_function_for_shap is defined. This function:

* Decodes the one-hot encoded input back into a nucleotide string.
* Calls the AlphaGenome API (dna_model.predict_sequence) for each sequence.
* Returns the sum of the values of the DNASE accessibility predictions for the K562 cell line.

Explainer Setup: A shap.KernelExplainer is initialized with the prediction function and the background data sample.

SHAP Calculation: explainer.shap_values(dna_flat) is called to calculate the SHAP values for the target 256-base sequence (also padded and one-hot encoded).

Result Processing: The resulting flat SHAP array (≈8192 values) is reshaped and summed across the 4 bases to yield a single contribution value per nucleotide position (shap_valores), focused only on the central 256-base region.

4. **Visualization**
Plotting: Both the ISM Alphagenome contributions (resultado_alphagenome) and the SHAP contributions (shap_valores) are plotted on the same graph for direct comparison.


In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Mon Oct  6 17:06:12 2025

@author: ajoven10@gmail.com
(I used gemini and copilot)
"""
from IPython.display import clear_output
! pip install alphagenome
clear_output()

In [ ]:
! pip install Bio
clear_output()


## Imports

In [ ]:
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
from alphagenome.data import track_data
import matplotlib.pyplot as plt
import pandas as pd
import os
from Bio import SeqIO
import numpy as np
import shap
import random
import gzip
import shutil
from IPython.display import display, HTML

In [ ]:
dna_model = dna_client.create(colab_utils.get_api_key())

## Previously

The hg38 genome is downloaded:

In [ ]:
url = "https://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/latest/hg38.fa.gz"
!wget {url}

archivo_gz="hg38.fa.gz"
archivo_fasta="hg38.fa"

with gzip.open(archivo_gz, 'rb') as f_in, open(archivo_fasta, 'wb') as f_out:
    shutil.copyfileobj(f_in, f_out)

# Remove the compressed file
os.remove(archivo_gz)

clear_output()

In [ ]:
def carga_genome():
    hg38_fasta = "/content/hg38.fa"  # Ruta al archivo FASTA
    genome = SeqIO.to_dict(SeqIO.parse(hg38_fasta, "fasta"))
    return(genome)

genoma = carga_genome()

## Working with alphagenome

Obtaining the relative nucleotide importance track for a 'chr19', 41003226, 41005274 with Alphagenome.

The steps from the [Aphagenome quick start notebook](https://www.alphagenomedocs.com/colabs/quick_start.html) are followed.

In [ ]:
# 2KB DNA sequence to use as context when making predictions.
sequence_interval = genome.Interval('chr19', 41003226, 41005274)
sequence_interval = sequence_interval.resize(dna_client.SEQUENCE_LENGTH_2KB)

# Mutate all bases in the central 500-base region of the sequence_interval.
ism_interval = sequence_interval.resize(500)

dnase_variant_scorer = variant_scorers.CenterMaskScorer(
    requested_output=dna_client.OutputType.RNA_SEQ,
    width=501,
    aggregation_type=variant_scorers.AggregationType.DIFF_MEAN,
)

variant_scores = dna_model.score_ism_variants(
    interval=sequence_interval,
    ism_interval=ism_interval,
    variant_scorers=[dnase_variant_scorer],
)

def extract_gen(adata):
  mask = (adata.var['ontology_curie'] == 'UBERON:0001114') & (adata.var['strand'] == '+')
  values = adata.X[:, mask]
  assert values.size == 1
  return values.flatten()[0]

ism_result = ism.ism_matrix(
    [extract_gen(x[0]) for x in variant_scores],
    variants = [v[0].uns['variant'] for v in variant_scores],
)


Now, I sum the resulting ISM value by position.

In [ ]:
resultado_alphagenome = np.sum(ism_result, axis=1)

## Working with Shap

In this code snippet, the DNA sequence is extracted from the previously downloaded genome, and 31222-bp  sequences from chromosome chr19 will be obtained to provide background for SHAP.

In [ ]:
from alphagenome.data import track_data, genome

gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_longest_transcript(gtf_transcripts)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

longest_transcripts = transcript_extractor.extract(sequence_interval) # This line is not needed here

In [ ]:
CROMOSOMA = 'chr19'
LONGITUD_SECUENCIA = 500  # len ism_interval

# Filter genes on the specified chromosome
genes_on_chromosome = gtf_transcripts[gtf_transcripts['Chromosome'] == CROMOSOMA]

secuencias_aleatorias = []

# Generate sequences starting 250 bp before each gene
for index, row in genes_on_chromosome.iterrows():
    gene_start = row['Start']
    # Calculate the start position 250 bp before the gene, ensuring it's not negative
    start = max(0, gene_start - 250)
    end = start + LONGITUD_SECUENCIA

    # Ensure the end position is within the chromosome length
    if end <= len(genoma[CROMOSOMA].seq):
      sub_secuencia = str(genoma[CROMOSOMA].seq[start:end]).upper()
      secuencias_aleatorias.append(sub_secuencia)

# Sample 1000 sequences if there are more than 1000
if len(secuencias_aleatorias) > 1000:
    secuencias_aleatorias = random.sample(secuencias_aleatorias, 1000)

# The sequences are resized to a length of 2048 bp (base pairs).
registros_padded = [secu.center(2048, 'N') for secu in secuencias_aleatorias]

NUM_SECUENCIAS = len(secuencias_aleatorias) # Update NUM_SECUENCIAS  generated
print(f"Generated {NUM_SECUENCIAS} sequences based on genes in {CROMOSOMA}.")

# Delete variables that are no longer needed
del gtf
del gtf_transcripts

The sequence defined by the ism_interval object is extracted from the genome for processing.

In [ ]:
dna = str(genoma[ism_interval.chromosome].seq[ism_interval.start:ism_interval.end]).upper()
secuencia = str(genoma[sequence_interval.chromosome].seq[sequence_interval.start:sequence_interval.end]).upper()

The necessary functions are defined to convert a DNA sequence to the one-hot encoding required for calls to the SHAP functions.

In [ ]:
def one_hot_encode_multiple_sequences(sequences):
    num_sequences = len(sequences)
    one_hot_array = np.zeros((num_sequences, len(sequences[0]), 4), dtype=np.int8)
    for i, seq in enumerate(sequences):
        for j, char in enumerate(seq):
            if char == "A" or char == "a": one_hot_array[i, j, 0] = 1
            elif char == "C" or char == "c": one_hot_array[i, j, 1] = 1
            elif char == "G" or char == "g": one_hot_array[i, j, 2] = 1
            elif char == "T" or char == "t": one_hot_array[i, j, 3] = 1
    return one_hot_array

def one_hot_encode_single_sequence(sequence):
    return one_hot_encode_multiple_sequences([sequence])


Define the prediction function for SHAP". This function needs to accept a data array (in one-hot format) and output the model's predictions.

In [ ]:
call_counter = 0

In [ ]:
def predict_function_for_shap(one_hot_sequences_flat):
    """
    Prediction function for SHAP. It decodes the flat array, calls the API, and returns the prediction value.
    """
    global call_counter
    num_sequences = one_hot_sequences_flat.shape[0]
    one_hot_sequences_3d = one_hot_sequences_flat.reshape(num_sequences, dna_len, 4)

    predictions = []
    for one_hot_seq_3d in one_hot_sequences_3d:
        nucleotide_string = "".join(
            "A" if np.all(v == [1, 0, 0, 0]) else
            "C" if np.all(v == [0, 1, 0, 0]) else
            "G" if np.all(v == [0, 0, 1, 0]) else
            "T" if np.all(v == [0, 0, 0, 1]) else
            "N"
            for v in one_hot_seq_3d
        )

        result = dna_model.predict_sequence(
            sequence=nucleotide_string,
            requested_outputs=[dna_client.OutputType.RNA_SEQ],
            ontology_terms=ontology
        )
        # predictions.append(result.dnase.values[secuencia_len // 2])
        # predictions.append(np.sum(result.rna_seq.values[774:1274]))
        predictions.append(np.sum(result.rna_seq.values))
        # predictions.append(np.sum(np.abs(result.dnase.values)))
        call_counter += 1

    return np.array(predictions)

The sequence to be explained is prepared.

In [ ]:
dna_len = 2048
ontology = ['UBERON:0001114']
dna_padded = dna.center(2048, 'N')

# Encode the sequence to be explained and flatten it.
dna_one_hot = one_hot_encode_single_sequence(dna_padded)
dna_flat = dna_one_hot.reshape(1, -1)

K = 5 # background for shap

background_data_3d = one_hot_encode_multiple_sequences(registros_padded)
background_data_flat = background_data_3d.reshape(len(secuencias_aleatorias), -1)
background_sample_data = shap.kmeans(background_data_flat, K)
background_data_sample = background_sample_data.data


Ejecución de shap: secuencia objetivo y background:

In [ ]:
# Initizlize KernelExplainer (sin mascarador)
print("Calculando valores SHAP. Esto puede tomar tiempo...")
explainer = shap.KernelExplainer(predict_function_for_shap, background_data_sample)
shap_values_flat = explainer.shap_values(dna_flat)
print(f"El modelo de AlphaGenome fue accedido {call_counter} veces.")

In [ ]:
# 6. Reorganizar y visualizar los valores
# El resultado de shap_values_flat tendrá una forma de (1, 8192)
# Por lo tanto, necesitamos reformar el array para sumar los valores por posición
shap_values_3d = shap_values_flat.reshape(1, -1, 4)
# Ahora, suma los valores SHAP de cada base para cada posición
shap_values_summed = shap_values_3d.sum(axis=-1).squeeze()
shap_valores = shap_values_summed[774:1274]


In [ ]:
result = dna_model.predict_sequence(
    sequence=secuencia,
    requested_outputs=[dna_client.OutputType.RNA_SEQ],
    ontology_terms=ontology
)

## Plots

In [ ]:
metadata=result.rna_seq.metadata
intervalo_plot = genome.Interval('chr19', 41004000, 41004500)

rnaseq_track = track_data.TrackData(
    values=result.rna_seq.values,
    metadata=metadata,
    interval=sequence_interval # Set the interval explicitly during creation
)

# Corrected ism_metadata creation to match the shape of ism_result (500, 4)
ism_metadata = pd.DataFrame({
    'name': ['A', 'C', 'G', 'T'],
    'strand': ['.', '.', '.', '.']
})

ism_track_data = track_data.TrackData(
    values=ism_result, # ism_result should be (length, 4)
    metadata=ism_metadata, # Use the new metadata
    interval=ism_interval
)
#añadir track con valores shap
shap_metadata = rnaseq_track.filter_to_positive_strand().metadata
shap_values = np.expand_dims(shap_valores, axis=-1)
track_shap = track_data.TrackData(
    values=shap_values,
    metadata = shap_metadata,
    interval=ism_interval,
    )

plot_components.plot(
    components=[
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.Tracks(rnaseq_track.filter_to_positive_strand()),
        plot_components.Tracks(track_shap),
        plot_components.SeqLogo(scores=ism_track_data.values, scores_interval=ism_track_data.interval)
    ],
    interval= intervalo_plot
)

plt.show()

In [ ]:
top_10_alphagenome_contribuciones = np.argsort(resultado_alphagenome)[-20:]
top_10_shap_indices = np.argsort(shap_valores)[-10:]

print("Top 10 indices for Alphagenome (ISM):", top_10_alphagenome_contribuciones)
print("Top 10 indices for SHAP:", top_10_shap_indices)

common_indices = np.intersect1d(top_10_alphagenome_contribuciones, top_10_shap_indices)

print("Common indices in the top 10 for both methods:", common_indices)

In [ ]:
top_20_alphagenome_valores = np.argsort(resultado_alphagenome)[-20:]
common_indices = np.intersect1d(top_10_alphagenome_contribuciones, top_20_alphagenome_valores)
print("Common indices in the top 10 for both:", common_indices)


In [ ]:
dna_segment = dna

formatted_dna = []

for i in range(len(dna_segment)):
    index_in_ism = i
    nucleotide = dna_segment[i]
    color = 'black' # Default color
    font_size = '70%' # Default font size

    in_contribuciones = index_in_ism in top_10_alphagenome_contribuciones
    in_valores = index_in_ism in top_20_alphagenome_valores

    if in_contribuciones and in_valores:
        color = 'purple'
        font_size = '170%'
    elif in_contribuciones:
        # If only in top_10_alphagenome_contribuciones, highlight in red
        color = 'red'
        font_size = '170%'
    elif in_valores:
        # If only in top_10_alphagenome_valores, highlight in blue
        color = 'blue'
        font_size = '170%'

    # Apply HTML formatting
    formatted_nucleotide = f'<span style="color: {color}; font-size: {font_size};">{nucleotide}</span>'
    formatted_dna.append(formatted_nucleotide)

# Join the formatted nucleotides and display as HTML
html_output = "".join(formatted_dna)
display(HTML(f'<pre>{html_output}</pre>')) # Using <pre> to maintain spacing
